1. Постановка задачи - представлено в отчете

2. Анализ данных

In [ ]:
import numpy as np
import pandas as pd
import warnings

def perform_exploratory_data_analysis(npz_path, csv_path):
    raw_data = np.load(npz_path)
    X = raw_data['X']
    y = raw_data['y']
    df_meta = pd.read_csv(csv_path)

    n_samples, n_features = X.shape
    benign_count = np.sum(y == 0)
    malware_count = np.sum(y == 1)

    print(f"Число объектов: {n_samples:,}")
    print(f"Число признаков: {n_features}")
    print(f"Соотношение Benign (0): {benign_count:,} ({benign_count/n_samples*100:.2f}%)")
    print(f"Соотношение Malware (1): {malware_count:,} ({malware_count/n_samples*100:.2f}%)")

    df_meta['label'] = y

    df_meta['timestamp'] = pd.to_datetime(df_meta['timestamp'], format='mixed', utc=True)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        df_meta['year_month'] = df_meta['timestamp'].dt.to_period('M')

    malware_df = df_meta[df_meta['label'] == 1]
    top_families = malware_df['family'].value_counts()

    print(f"\nОбщее количество уникальных семейств малвари: {malware_df['family'].nunique()}")
    print("Топ-10 доминирующих семейств вредоносного ПО:")
    print(top_families.head(10))

    share_top_10 = top_families.head(10).sum() / len(malware_df) * 100
    print(f"-> Вывод: Топ-10 семейств покрывают {share_top_10:.2f}% всей малвари.")

    hash_col = None
    for col in df_meta.columns:
        if 'sha' in col.lower() or 'hash' in col.lower():
            hash_col = col
            break

    if hash_col:
        sha_duplicates = df_meta[hash_col].duplicated().sum()
        print(f"\nКоличество полных дубликатов файлов по колонке '{hash_col}': {sha_duplicates}")
    else:
        sha_duplicates = df_meta.duplicated(subset=['timestamp', 'family']).sum()
        print(f"Количество дубликатов по совпадению (timestamp + family): {sha_duplicates}")

    return df_meta, X, y

df_meta, X, y = perform_exploratory_data_analysis('bodmas.npz', 'bodmas_metadata.csv')


Число объектов: 134,435
Число признаков: 2381
Соотношение Benign (0): 77,142 (57.38%)
Соотношение Malware (1): 57,293 (42.62%)

Общее количество уникальных семейств малвари: 582
Топ-10 доминирующих семейств вредоносного ПО:
family
sfone       4729
wacatac     4694
upatre      3901
wabot       3673
small       3339
ganelp      2232
dinwod      2057
mira        1960
berbew      1749
sillyp2p    1616
Name: count, dtype: int64
-> Вывод: Топ-10 семейств покрывают 52.28% всей малвари.

Количество полных дубликатов файлов по колонке 'sha': 0


3. Протокол валидации

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

def run_validation_protocols(X, y, df_meta):
    model_params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'n_estimators': 100,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    print("\n--- Вариант 1: Случайное разбиение (80% Train / 20% Test) ---")
    X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf_rnd = lgb.LGBMClassifier(**model_params)
    clf_rnd.fit(X_train_rnd, y_train_rnd)
    preds_rnd = clf_rnd.predict(X_test_rnd)

    print("Результаты Случайного разбиения:")
    print(classification_report(y_test_rnd, preds_rnd, digits=4))


    print("\n--- Вариант 2: Временное разбиение (Хронологический сплит) ---")
    sort_indices = df_meta['timestamp'].argsort()
    X_sorted = X[sort_indices]
    y_sorted = y[sort_indices]
    df_meta_sorted = df_meta.iloc[sort_indices].reset_index(drop=True)

    split_idx = int(len(X_sorted) * 0.80)

    X_train_time = X_sorted[:split_idx]
    y_train_time = y_sorted[:split_idx]
    X_test_time = X_sorted[split_idx:]
    y_test_time = y_sorted[split_idx:]

    start_train = df_meta_sorted['timestamp'].dt.date.iloc[0]
    end_train = df_meta_sorted['timestamp'].dt.date.iloc[split_idx-1]
    start_test = df_meta_sorted['timestamp'].dt.date.iloc[split_idx]
    end_test = df_meta_sorted['timestamp'].dt.date.iloc[-1]

    print(f"Обучение: с {start_train} по {end_train}")
    print(f"Тестирование: с {start_test} по {end_test}")

    clf_time = lgb.LGBMClassifier(**model_params)
    clf_time.fit(X_train_time, y_train_time)
    preds_time = clf_time.predict(X_test_time)

    print("\nРезультаты Временного разбиения:")
    print(classification_report(y_test_time, preds_time, digits=4))

    f1_rnd = f1_score(y_test_rnd, preds_rnd)
    f1_time = f1_score(y_test_time, preds_time)
    print(f"F1-Score при случайном разбиении: {f1_rnd:.4f}")
    print(f"F1-Score при временном разбиении: {f1_time:.4f}")
    print(f"Реальное падение эффективности из-за дрейфа данных: {(f1_rnd - f1_time)*100:.2f}%")

run_validation_protocols(X, y, df_meta)



--- Вариант 1: Случайное разбиение (80% Train / 20% Test) ---


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Результаты Случайного разбиения:
              precision    recall  f1-score   support

           0     0.9965    0.9970    0.9968     15428
           1     0.9960    0.9953    0.9956     11459

    accuracy                         0.9963     26887
   macro avg     0.9962    0.9962    0.9962     26887
weighted avg     0.9963    0.9963    0.9963     26887


--- Вариант 2: Временное разбиение (Хронологический сплит) ---
Обучение: с 2007-01-01 по 2020-06-30
Тестирование: с 2020-06-30 по 2020-09-30


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Результаты Временного разбиения:
              precision    recall  f1-score   support

           0     0.9930    0.9991    0.9961     13152
           1     0.9991    0.9933    0.9962     13735

    accuracy                         0.9961     26887
   macro avg     0.9961    0.9962    0.9961     26887
weighted avg     0.9962    0.9961    0.9961     26887

F1-Score при случайном разбиении: 0.9956
F1-Score при временном разбиении: 0.9962
Реальное падение эффективности из-за дрейфа данных: -0.06%


4. Baseline и основные модели

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier


sort_indices = df_meta['timestamp'].argsort()
X_sorted = X[sort_indices]
y_sorted = y[sort_indices]

split_idx = int(len(X_sorted) * 0.80)
X_train, X_test = X_sorted[:split_idx], X_sorted[split_idx:]
y_train, y_test = y_sorted[:split_idx], y_sorted[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {}
preds = {}
probs = {}


print("Обучение Dummy Baseline...")
models['Dummy'] = DummyClassifier(strategy='most_frequent')
models['Dummy'].fit(X_train, y_train)
preds['Dummy'] = models['Dummy'].predict(X_test)
probs['Dummy'] = models['Dummy'].predict_proba(X_test)[:, 1]


print("Обучение Logistic Regression...")
models['LogReg'] = LogisticRegression(C=1.0, max_iter=500, solver='saga', random_state=42, n_jobs=-1)
models['LogReg'].fit(X_train_scaled, y_train)
preds['LogReg'] = models['LogReg'].predict(X_test_scaled)
probs['LogReg'] = models['LogReg'].predict_proba(X_test_scaled)[:, 1]


print("Обучение Random Forest...")
models['RandomForest'] = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
models['RandomForest'].fit(X_train, y_train)
preds['RandomForest'] = models['RandomForest'].predict(X_test)
probs['RandomForest'] = models['RandomForest'].predict_proba(X_test)[:, 1]


print("Обучение LightGBM...")
models['LightGBM'] = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1)
models['LightGBM'].fit(X_train, y_train)
preds['LightGBM'] = models['LightGBM'].predict(X_test)
probs['LightGBM'] = models['LightGBM'].predict_proba(X_test)[:, 1]


print("Обучение MLP (Нейросеть)...")
models['MLP'] = MLPClassifier(hidden_layer_sizes=(64,), max_iter=20, random_state=42, early_stopping=True)
models['MLP'].fit(X_train_scaled, y_train)
preds['MLP'] = models['MLP'].predict(X_test_scaled)
probs['MLP'] = models['MLP'].predict_proba(X_test_scaled)[:, 1]


Обучение Dummy Baseline...
Обучение Logistic Regression...
Обучение Random Forest...
Обучение LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Обучение MLP (Нейросеть)...


5. Метрики и цена ошибок

In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)

def evaluate_security_models(y_true, preds_dict, probs_dict):
    metrics_report = []

    for model_name in preds_dict.keys():
        y_pred = preds_dict[model_name]
        y_prob = probs_dict[model_name]


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_true, y_prob)
        pr_auc = average_precision_score(y_true, y_prob)

        metrics_report.append({
            'Модель': model_name,
            'FP (Ложные алерты)': fp,
            'FN (Пропуски малвари)': fn,
            'Precision (Точность)': round(precision, 4),
            'Recall (Полнота)': round(recall, 4),
            'F1-Score': round(f1, 4),
            'ROC-AUC': round(roc_auc, 4),
            'PR-AUC': round(pr_auc, 4)
        })

    df_report = pd.DataFrame(metrics_report)
    print(df_report.to_string(index=False))


evaluate_security_models(y_test, preds, probs)


      Модель  FP (Ложные алерты)  FN (Пропуски малвари)  Precision (Точность)  Recall (Полнота)  F1-Score  ROC-AUC  PR-AUC
       Dummy                   0                  13735                0.0000            0.0000    0.0000   0.5000  0.5108
      LogReg                  62                    374                0.9954            0.9728    0.9839   0.9955  0.9967
RandomForest                  12                    390                0.9991            0.9716    0.9852   0.9997  0.9997
    LightGBM                  13                    130                0.9990            0.9905    0.9948   0.9999  0.9999
         MLP                  26                    473                0.9980            0.9656    0.9815   0.9973  0.9974


6. Выбор порога принятия решения

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, confusion_matrix

def explore_thresholds(y_true, y_probs):

    thresholds = np.linspace(0.01, 0.99, 99)
    records = []

    for t in thresholds:
        y_pred = (y_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)

        records.append({'Threshold': t, 'FP': fp, 'FN': fn, 'Precision': p, 'Recall': r})

    df_t = pd.DataFrame(records)

    auto_mode = df_t[df_t['FP'] <= 5].sort_values(by='Recall', ascending=False).iloc[0]

    f1_scores = 2 * (df_t['Precision'] * df_t['Recall']) / (df_t['Precision'] + df_t['Recall'] + 1e-8)
    best_f1_idx = f1_scores.idxmax()
    balance_mode = df_t.iloc[best_f1_idx]

    hunting_mode = df_t[df_t['FN'] <= 15].sort_values(by='Precision', ascending=False).iloc[0]

    print(f"1. Режим автоблокировки (Высокое доверие): Порог = {auto_mode['Threshold']:.2f}")
    print(f"   Ложные срабатывания (FP): {int(auto_mode['FP'])}, Пропуски (FN): {int(auto_mode['FN'])}")
    print(f"2. Режим приоритизации (Сбалансированный): Порог = {balance_mode['Threshold']:.2f}")
    print(f"   Ложные срабатывания (FP): {int(balance_mode['FP'])}, Пропуски (FN): {int(balance_mode['FN'])}")
    print(f"3. Режим Threat Hunting (Низкое доверие):    Порог = {hunting_mode['Threshold']:.2f}")
    print(f"   Ложные срабатывания (FP): {int(hunting_mode['FP'])}, Пропуски (FN): {int(hunting_mode['FN'])}")

explore_thresholds(y_test, probs['LightGBM'])


1. Режим автоблокировки (Высокое доверие): Порог = 0.78
   Ложные срабатывания (FP): 5, Пропуски (FN): 465
2. Режим приоритизации (Сбалансированный): Порог = 0.18
   Ложные срабатывания (FP): 38, Пропуски (FN): 39
3. Режим Threat Hunting (Низкое доверие):    Порог = 0.01
   Ложные срабатывания (FP): 612, Пропуски (FN): 2


7. Анализ ошибок и интерпретируемость

In [ ]:
def analyze_feature_importance(model, feature_names_count=2381):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    print("Топ-10 наиболее влиятельных индексов признаков EMBER для детекции:")
    for i in range(10):
        print(f"   Ранг {i+1}: Признак #{indices[i]} (Важность: {importances[indices[i]]})")

analyze_feature_importance(models['LightGBM'])


Топ-10 наиболее влиятельных индексов признаков EMBER для детекции:
   Ранг 1: Признак #626 (Важность: 125)
   Ранг 2: Признак #2355 (Важность: 119)
   Ранг 3: Признак #637 (Важность: 104)
   Ранг 4: Признак #655 (Важность: 101)
   Ранг 5: Признак #2359 (Важность: 77)
   Ранг 6: Признак #32 (Важность: 68)
   Ранг 7: Признак #685 (Важность: 63)
   Ранг 8: Признак #2364 (Важность: 62)
   Ранг 9: Признак #2353 (Важность: 61)
   Ранг 10: Признак #2356 (Важность: 60)


8. Проверка устойчивости

In [ ]:
def stress_test_model(model, X_val, y_val, noise_level=0.1):
    base_preds = model.predict(X_val)
    base_f1 = f1_score(y_val, base_preds)

    noise = np.random.normal(0, noise_level, X_val.shape)
    X_noisy = X_val + noise

    noisy_preds = model.predict(X_noisy)
    noisy_f1 = f1_score(y_val, noisy_preds)

    print(f"F1-Score на чистом временном тесте: {base_f1:.4f}")
    print(f"F1-Score при зашумлении признаков (Уровень шума {noise_level}): {noisy_f1:.4f}")
    print(f"Деградация качества: {(base_f1 - noisy_f1)*100:.2f}%")

stress_test_model(models['LightGBM'], X_test, y_test, noise_level=0.15)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


F1-Score на чистом временном тесте: 0.9948
F1-Score при зашумлении признаков (Уровень шума 0.15): 0.6682
Деградация качества: 32.65%


9. Анализ рисков безопасности ML-компонента - в отчете

10. Итоговое заключение о надежности - в отчете